In [ ]:
# 本文件用于在日度数据层面新增日度的特征变量，以便后面形成月度截面面板
%run config_0.ipynb
%run data_io_1.ipynb
%run clean_and_universe_2.ipynb

import pandas as pd
import numpy as np

In [ ]:
# 市值特征，可用于规模因子smb，以及市值加权处理的排序
def get_market_feature(df):
    df[day_market_value] = df[close_price] * df[total_shares]
    return df


In [ ]:
# 超额收益率特征，可用于市场因子mkt和动量因子mom
def get_excess_return(df):
    df[day_excess_ret] = df[daily_ret] - df[daily_rfret]
    df[w] = df.groupby(date)[day_market_value].transform(lambda x:x / x.sum())
    print(df[w].head())
    print(df[w].shape)
    print(df[day_excess_ret].head())
    print(df[day_excess_ret].shape)
    return df


def get_momentum(df):
    formation_month = df[date].dt.to_period('M').rename('formation_month')
    monthly_table = pd.DataFrame({
        stock_code: df[stock_code].to_numpy(),
        'formation_month': formation_month.to_numpy(),
        'monthly_gross_return': 1 + df[daily_ret].to_numpy(),
    })
    monthly_table = (
        monthly_table
        .groupby([stock_code, 'formation_month'], as_index=False)
        ['monthly_gross_return']
        .prod()
        .sort_values([stock_code, 'formation_month'])
        .reset_index(drop=True)
    )
    monthly_table['rolling_gross_return'] = (
        monthly_table.groupby(monthly_table[stock_code])
        ['monthly_gross_return']
        .transform(
            lambda stock_returns: stock_returns
            .rolling(11, min_periods=11)
            .apply(np.prod, raw=True)
        )
    )
    monthly_momentum = (
        monthly_table.groupby(monthly_table[stock_code])
        ['rolling_gross_return'].shift(1) - 1
    )
    monthly_keys = list(
        zip(monthly_table[stock_code], monthly_table['formation_month'])
    )
    monthly_momentum = pd.Series(
        monthly_momentum.to_numpy(), index=monthly_keys
    )
    momentum_keys = pd.Series(
        list(zip(df[stock_code], formation_month)), index=df.index
    )
    df[day_mom] = momentum_keys.map(monthly_momentum)
    return df


In [ ]:
# 市净率bm特征，可用于价值因子hml
def get_bm(df):
    df[day_bm] = 1 / df[pb]
    return df

In [ ]:
# 盈利收益率EP特征，可用于CH3价值因子vmg
def get_ep(df):
    df[day_ep] = df[eps] / df[close_price]
    return df   

In [ ]:
# 现金流价格比 CP：CENetIncr / 当日总市值。
# 年度现金流事件与日度对齐由 clean_and_universe_2 的 prepare_annual_cash_flow / align_annual_value 提供。
def get_cp(df, cf_df):
    require_columns(df, [stock_code, date, day_market_value], '日股表')
    if df.empty:
        df[day_cp] = pd.Series(index=df.index, dtype='float64')
        return df

    # 分子：把最近一份年报的净现金流事件对齐回日度行，得到与 df 等长的年度现金流序列。
    # 事件构造与对齐见 clean_and_universe_2 的 prepare_annual_cash_flow / align_annual_value。
    numerator = align_annual_value(
        df, prepare_annual_cash_flow(cf_df)
    )
    # 分母：当日总市值（get_market_feature 已算好），转成 numpy 数组以便按位置做布尔除法。
    denominator = df[day_market_value].to_numpy(
        dtype='float64', copy=False
    )
    # 预填全 NaN 的 CP，只在“有限且为正”的市值位置上相除，0/负/无穷分母一律留 NaN。
    # 分子分母防御详见 learning/特征构造模型/step5_分子分母防御.md 与 get_cp对齐防御逐行详解.md。
    cp_values = np.full(len(df), np.nan, dtype='float64')
    valid_denominator = np.isfinite(denominator) & (denominator > 0)
    cp_values[valid_denominator] = (
        numerator[valid_denominator] / denominator[valid_denominator]
    )
    df[day_cp] = cp_values
    return df

In [ ]:
# 净资产收益率roe使用原始资产负债表中的即可，可用于盈利因子构造

In [ ]:
# 资产增长：本期年报总资产 / 上一期年报总资产。
# 年度资产增长事件与日度对齐由 clean_and_universe_2 的 prepare_annual_asset_growth / align_annual_value 提供。
def get_asset_growth(df, bs_df):
    require_columns(df, [stock_code, date], '日股表')
    if df.empty:
        df[day_asset_growth] = pd.Series(
            index=df.index, dtype='float64'
        )
        return df

    df[day_asset_growth] = align_annual_value(
        df, prepare_annual_asset_growth(bs_df)
    )
    return df

In [ ]:
# 一个月波动率：过去 20 个交易日日收益的标准差，可用于构造波动率异象（volatility）
def get_one_month_volatility(df):
    df[day_volatility] = (
        df[[stock_code , date , daily_ret]]
        .sort_values([stock_code , date])
        .groupby(stock_code)[daily_ret]
        .rolling(20 , min_periods = 20)
        .std()
        .reset_index(level = 0 , drop = True)
        .reindex(df.index)
    )
    return df

In [ ]:
# MAX：过去 20 个交易日的最高日收益，可用于构造波动率异象（volatility）
def get_max_return(df):
    df[day_max] = (
        df[[stock_code , date , daily_ret]]
        .sort_values([stock_code , date])
        .groupby(stock_code)[daily_ret]
        .rolling(20 , min_periods = 20)
        .max()
        .reset_index(level = 0 , drop = True)
        .reindex(df.index)
    )
    return df

In [ ]:
# Accruals（应计利润），用于构造应计项目（Accruals）
#   Accrual = (ΔCA − ΔCash) − (ΔCL − ΔSTD − ΔTP) − Dep
#   年度值由 clean_and_universe_2 的 prepare_annual_accruals 构造（跨年差分在年度层完成），
#   这里只做向后对齐；口径与验证见 learning/特征构造模型/应计利润与净经营资产构造详解.md。
def get_accruals(df, bs_df, cf_df, fas_df):
    require_columns(df, [stock_code, date], '日股表')
    if df.empty:
        df[day_accrual] = pd.Series(index=df.index, dtype='float64')
        return df

    df[day_accrual] = align_annual_value(
        df, prepare_annual_accruals(bs_df, cf_df, fas_df)
    )
    return df

In [ ]:
# NOA（净经营资产），用于构造应计项目（Accruals）
#   NOA_t = (OA_t − OL_t) / TotalAssets_{t-1}
#     OA = 总资产 − 货币资金 − 交易性金融资产
#     OL = 总负债 − 短期借款 − 长期借款（与论文定义化简等价）
#   年度值由 clean_and_universe_2 的 prepare_annual_noa 构造，这里只做向后对齐。
def get_noa(df, bs_df):
    require_columns(df, [stock_code, date], '日股表')
    if df.empty:
        df[day_noa] = pd.Series(index=df.index, dtype='float64')
        return df

    df[day_noa] = align_annual_value(
        df, prepare_annual_noa(bs_df)
    )
    return df

In [ ]:
# Illiquidity（非流动性），用于构造流动性因子（Illiquidity）
#   Amihud(2002)：Illiq_t = |ret_t| / volume_t ，volume_t 为当日美元成交额(Trdsum)
#   因子 = 过去 20 个交易日 Illiq 的平均值

# 下面 _roll_by_stock 是 cell 13 / 14 / 15 共用的「按股票滚 N 日」辅助函数
def _roll_by_stock(df , values , window , how = 'mean'):
    tmp = df[[stock_code , date]].copy()          # 保留原 index
    tmp['_v'] = values
    tmp = tmp.sort_values([stock_code , date])
    rolled = tmp.groupby(stock_code)['_v'].rolling(window , min_periods = window)
    if how == 'mean':
        rolled = rolled.mean()
    elif how == 'std':
        rolled = rolled.std()
    elif how == 'max':
        rolled = rolled.max()
    else:
        raise ValueError('how 只支持 mean / std / max')
    return rolled.reset_index(level = 0 , drop = True).reindex(df.index)

def get_illiquidity(df):
    illiq = df[daily_ret].abs() / df[trade_amount]
    df[day_illiquidity] = _roll_by_stock(df , illiq , 20 , 'mean')
    return df

In [ ]:
# Twelve_month_turnover（12月换手率），用于构造换手率因子（Turnover）
#   日换手率 = 成交股数(Trdvol) / 总股本(Fullshr)
#   因子 = 过去 250 个交易日日换手率的平均值
def get_turnover_12m(df):
    turnover = df[trade_volumn] / df[total_shares]
    df[day_turnover_12m] = _roll_by_stock(df , turnover , 250 , 'mean')
    return df

In [ ]:
# One_month_abnormal_turnover（1月异常换手率），用于构造换手率因子（Turnover）
#   因子 = 过去 20 日平均日换手率 / 过去 250 日平均日换手率
def get_abn_turnover_1m(df):
    turnover = df[trade_volumn] / df[total_shares]
    to20  = _roll_by_stock(df , turnover , 20 , 'mean')
    to250 = _roll_by_stock(df , turnover , 250 , 'mean')
    df[day_abn_turnover_1m] = to20 / to250
    return df

In [ ]:
# 反转异象用monthly_return排序即可构造

In [ ]:
# 独立运行 features_3 时执行完整演示流程；被其他 Notebook 导入时可暂时关闭。
if globals().get('run_features_pipeline', True):
    df = read_daily_data()
    df = cleaning(df)
    df = universe(df)
    df = get_market_feature(df)
    df = get_excess_return(df)
    df = get_momentum(df)
    print('加入日度特征后的数据表示例')
    print(df.head())
    print('加入日度特征后的数据表形状')
    print(df.shape)
